# Local learning rules

The code of [sparxml.dev/learn/local-rules](https://sparxml.dev/learn/local-rules/), run on a CPU runtime. The first cell installs sparx.

In [ ]:
%pip install -q 'sparxml[nir] @ git+https://github.com/AshishKumar4/sparx'

In [ ]:
import jax

from sparx.dynamics import (
                            Delta,
                            LeakyIntegrateAndFire,
                            PairSTDP,
                            Receptor,
)
from sparx.graph import (
                            FixedProbability,
                            Network,
                            PoissonInput,
                            Population,
                            Projection,
                            simulate,
)

# NEST's stdp_synapse with additive bounds
stdp = PairSTDP(tau_plus=16.8, tau_minus=33.7, lambda_=0.005,
                alpha=0.55, mu_plus=0.0, mu_minus=0.0, w_max=1.0)
delta = {"ampa": Receptor(Delta())}
listener = LeakyIntegrateAndFire(tau_m=10.0, e_l=0.0, v_th=20.0,
                                 v_reset=0.0, t_ref=1.0)
network = Network(
    populations=(Population("in", 400, LeakyIntegrateAndFire(), delta),
                 Population("out", 1, listener, delta)),
    projections=(Projection("in", "out", FixedProbability(1.0),
                            weight=0.475, delay=1.0, receptor="ampa",
                            plasticity=stdp),),
    inputs=(PoissonInput("in", rate=40.0, weight=30.0,
                         receptor="ampa"),),
    dt=1.0,
)
result = simulate(network, network.init(jax.random.key(0)),
                  duration=2000.0, key=jax.random.key(1))
connections = network.connections(result.variables)
weight = connections["in->out:ampa"].weight
print(weight.min(), weight.max())    # as STDP left them

In [ ]:
import jax
import jax.numpy as jnp

from sparx.dynamics import LIFCell, decay
from sparx.learn import EPropParams, bptt_loss, eprop
from sparx.surrogate import Triangle

keys = jax.random.split(jax.random.key(0), 4)
T, inputs, size = 200, 20, 50
u = (jax.random.uniform(keys[0], (T, 1, inputs)) < 0.05) * 1.0
t = jnp.arange(T)
target = jnp.sin(2 * jnp.pi * t / 100)[:, None, None]
params = EPropParams(jax.random.normal(keys[1], (inputs, size)),
                     0.15 * jax.random.normal(keys[2], (size, size)),
                     0.05 * jax.random.normal(keys[3], (size, 1)),
                     jnp.zeros(1))
cell = LIFCell(decay(tau=20.0), surrogate=Triangle(scale=0.3),
               detach_reset=True)


def loss(y, target):
    return 0.5 * jnp.sum((y - target) ** 2)


_, online = eprop(cell, params, u, target, loss, tau=10.0)


def through_time(p, cut):
    return bptt_loss(cell, p, u, target, loss, tau=10.0,
                     cut_recurrence=cut)


full = jax.grad(through_time)(params, False)
cut = jax.grad(through_time)(params, True)


def cosine(a, b):
    a = jnp.concatenate([x.ravel() for x in a])
    b = jnp.concatenate([x.ravel() for x in b])
    return float(a @ b / jnp.linalg.norm(a) / jnp.linalg.norm(b))


print("e-prop vs BPTT:", cosine(online, full))
print("e-prop vs BPTT with the recurrence cut:", cosine(online, cut))